# Notebook 4 (Bonus): Fast Search with MMseqs2 and Foldseek

**Chem 274A · Lab Week 11 · Sequence Similarity Networks**

<div class="alert alert-block alert-info">
<h3>Learning goals</h3>

- Run command-line bioinformatics programs from Python with `subprocess`.
- See how a search tool's sensitivity setting changes what it finds.
- Rebuild the SSN from MMseqs2 bit scores, and repeat the `mystery_3` question with a structure search.
</div>

<div class="alert alert-block alert-info">
This notebook needs <b>MMseqs2</b> and <b>Foldseek</b>, which come from the <code>bioconda</code> channel (Linux, macOS and WSL only).
Create the bonus environment with <code>make environment-bonus</code>, activate it with
<code>conda activate chem274A_ssn_bonus</code>, and start Jupyter from there.
</div>

For decades the standard search tool has been **BLAST**, and EFI-EST still offers it. Newer tools do the same kind of search
much faster, which is what makes it possible to compare the hundreds of millions of sequences now in the databases:

- **[MMseqs2](https://github.com/soedinglab/MMseqs2)** quickly filters out pairs that share no similar short words, then
  runs a full alignment on the rest. It reports the same kinds of numbers as BLAST, including **E-values**
  (*how many hits this good you would expect by chance in a database this size*; smaller is better) and **bit scores**.
- **[Foldseek](https://github.com/steineggerlab/foldseek)** applies the same idea to 3D structures.

Both are command-line programs, so we run them from Python with the `subprocess` module.


In [ ]:
import shutil
import subprocess
from collections import Counter
from pathlib import Path

import networkx as nx
import pandas as pd

from ssn_plot import draw_ssn

meta = pd.read_csv("data/globins_metadata.csv").set_index("id")

for tool in ["mmseqs", "foldseek"]:
    if shutil.which(tool) is None:
        print(f"{tool} is not installed in this environment.")
        print("Run `make environment-bonus`, then `conda activate chem274A_ssn_bonus` and restart Jupyter.")
    else:
        print(tool, "found")

## Step 1: Run an all-vs-all search

`mmseqs easy-search` takes a query file, a target file, an output file and a temporary folder. For an all-vs-all
comparison, the query and the target are the same file.

`subprocess.run` takes the command as a **list of strings**, the same words you would type in a terminal.
`--format-output` lists the columns we want:

| Field | Meaning |
|---|---|
| `query` | query sequence ID |
| `target` | target (database hit) sequence ID |
| `fident` | fraction of identical residues (0–1, unlike BLAST's percent) |
| `alnlen` | alignment length |
| `evalue` | E-value |
| `bits` | bit score |

`-s` is the **sensitivity**: a higher number finds weaker matches but runs more slowly. 5.7 is the default.


In [ ]:
Path("search_results").mkdir(exist_ok=True)
columns = ["query", "target", "fident", "alnlen", "evalue", "bits"]

def run_mmseqs(output, sensitivity):
    subprocess.run(
        ["mmseqs", "easy-search", "data/globins.fasta", "data/globins.fasta", output, "search_results/tmp",
         "--format-output", ",".join(columns),
         "-e", "1e-3",
         "-s", str(sensitivity),
         "--max-seqs", "1000",
         "-v", "1"],
        check=True,
    )

run_mmseqs("search_results/default.tsv", sensitivity=5.7)
print(open("search_results/default.tsv").read()[:300])

### Loading the results

Look closely at the IDs above. MMseqs2 shortened `sp|P69905|HBA_HUMAN` to just `P69905`, so they no longer match the
index of `meta`. Real tools do things like this all the time, and the fix is usually a few lines of pandas.
`load_hits(path)` reads the file (it has no header row, so we pass `names=columns`), maps each accession back to
its full ID (mystery IDs stay as they are), and removes the rows where a sequence hit itself.

In [ ]:
accession_to_id = {row.accession: seq_id for seq_id, row in meta.iterrows() if pd.notna(row.accession)}

def load_hits(path):
    hits = pd.read_csv(path, sep="\t", names=columns)
    hits["query"] = hits["query"].replace(accession_to_id)
    hits["target"] = hits["target"].replace(accession_to_id)
    return hits[hits["query"] != hits["target"]]

hits_default = load_hits("search_results/default.tsv")
print(len(hits_default), "hits")

### Turning up the sensitivity

The next cell runs the search again at `-s 7.5` and compares it with the default. Does `mystery_3` appear as a query
in each search? What would you have concluded if you had only run the default?

In [ ]:
run_mmseqs("search_results/sensitive.tsv", sensitivity=7.5)
hits = load_hits("search_results/sensitive.tsv")

for label, table in [("default", hits_default), ("sensitive", hits)]:
    print(f"{label:10s} {len(table):5d} hits, mystery_3 is the query in {(table['query'] == 'mystery_3').sum()}")

### A bit-score SSN

`build_mmseqs_ssn(hits, min_bits)` builds the network the same way as `build_ssn`, with an edge for every hit whose bit
score is at least `min_bits`. The cell tries several cutoffs and draws the network at 150. Which cutoff gives one
cluster per protein type, like your choice in Notebook 3?

In [ ]:
def build_mmseqs_ssn(hits, min_bits):
    G = nx.Graph()
    for seq_id, row in meta.iterrows():
        G.add_node(seq_id, protein_type=row["protein_type"], organism=row["organism"])
    strong = hits[hits["bits"] >= min_bits]
    for query, target, bits in zip(strong["query"], strong["target"], strong["bits"]):
        G.add_edge(query, target, bits=bits)
    return G

for min_bits in [75, 100, 150, 175, 200]:
    G = build_mmseqs_ssn(hits, min_bits)
    comps = sorted(nx.connected_components(G), key=len, reverse=True)
    print(f"bits >= {min_bits}: {len(comps)} clusters")
    for c in comps[:7]:
        print("    ", len(c), dict(Counter(G.nodes[n]["protein_type"] for n in c)))

draw_ssn(build_mmseqs_ssn(hits, 150), title="MMseqs2 SSN, bit score ≥ 150");

## Things to know about bit scores and E-values

- Each hit is directional. `A → B` and `B → A` can get slightly different numbers, so here an edge appears if *either* direction passes.
- E-values depend on the **database size**. The same pair of proteins gets a *larger* (worse) E-value in a bigger database.
  That's why EFI-EST uses an "alignment score" based on the bit score, which doesn't change with database size.
- A bit score still grows with alignment length, so a fixed bit-score cutoff is not as length-independent as your normalized score.


## Step 2: Search by structure with Foldseek

In Notebook 3 you ran TM-align on `mystery_3` against each structure, one pair at a time. Foldseek searches a whole folder (or the entire
[AlphaFold Database](https://alphafold.ebi.ac.uk)) in one command. `alntmscore` is the TM-score of each alignment,
and `-e 10` keeps weak hits so that we can see the controls if they turn up.

You can also upload a structure to the [Foldseek web server](https://search.foldseek.com), which searches the
PDB and AlphaFold databases in seconds.


In [ ]:
fold_columns = ["target", "fident", "alnlen", "evalue", "alntmscore"]

subprocess.run(
    ["foldseek", "easy-search", "data/structures/mystery_3.pdb", "data/structures",
     "search_results/foldseek.tsv", "search_results/tmp_foldseek",
     "--format-output", ",".join(fold_columns),
     "-e", "10",
     "-v", "1"],
    check=True,
)

fold_hits = pd.read_csv("search_results/foldseek.tsv", sep="\t", names=fold_columns)
fold_hits[~fold_hits["target"].str.startswith("mystery_3")].sort_values("alntmscore", ascending=False)

<div class="alert alert-block alert-warning">
<h3>Compare the methods</h3>

1. Do Foldseek's top hits agree with your TM-align results from Notebook 3? Which structures are missing, and why?
2. Look at the `fident` column. Could a sequence-only tool have made these matches with the same confidence?
3. In Notebook 3 you estimated how long 10,000 sequences would take. Which tool would you use for a network that size,
   and what would you lose compared with checking every pair with `PairwiseAligner`?
</div>
